# PDF Library Organizer — Colab Edition

Runs entirely in your browser via Google Colab — no local Python install needed.

**What this does:**
1. Mounts your Google Drive
2. Scans a folder of PDF papers (your 1200+ papers folder)
3. Extracts the real title from inside each PDF
4. Classifies each paper with Gemini AI into 4 categories
5. Renames + moves files into matching subfolders
6. Exports a colour-coded Excel report

**Before you start:** get a free Gemini API key at https://aistudio.google.com (no credit card needed — sign in with Google, click "Create API key")

Run each cell below in order (Shift+Enter).

## Cell 1 — Install dependencies

In [ ]:
!pip install -q pypdf openpyxl google-generativeai
print("Dependencies installed.")

## Cell 2 — Mount Google Drive

A popup/link will appear — click it, sign in, copy the authorization code back here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## Cell 3 — Set your papers folder path

Find your folder in the left sidebar (folder icon) under `drive/MyDrive/...` and copy its path here.

Example: if your papers are in `MyDrive/MyResearch/AllPapers`, then:
```python
PAPER_DIR = "/content/drive/MyDrive/MyResearch/AllPapers"
```

In [ ]:
PAPER_DIR = "/content/drive/MyDrive/MyResearch/AllPapers"  # <-- EDIT THIS

from pathlib import Path
paper_path = Path(PAPER_DIR)
if not paper_path.exists():
    print(f"[ERROR] Folder not found: {paper_path}")
    print("Double-check the path above against the Files sidebar on the left.")
else:
    pdf_count = len(list(paper_path.glob('*.pdf')))
    print(f"Folder found: {paper_path}")
    print(f"PDF files at top level: {pdf_count}")

## Cell 4 — Enter your Gemini API key

Uses `getpass` so the key is hidden as you type and is **not** saved into the notebook file.

In [ ]:
import getpass
import google.generativeai as genai

GEMINI_API_KEY = getpass.getpass("Paste your Gemini API key and press Enter: ")
genai.configure(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-1.5-flash"
_gemini_client = genai.GenerativeModel(GEMINI_MODEL)
print("Gemini client ready.")

## Cell 5 — Core functions
(title extraction, classification, Excel report — same logic as the local script)

In [ ]:
import re
import shutil
import sys
import time
import unicodedata

RATE_LIMIT_SEC = 4.5  # ~13 RPM, free tier allows 15 RPM

def _sanitize_filename(title, max_len=120):
    title = unicodedata.normalize("NFKD", title)
    title = title.encode("ascii", "ignore").decode("ascii")
    title = re.sub(r'[\\/:*?"<>|]', "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title[:max_len] if title else ""

def extract_title(pdf_path):
    import pypdf
    try:
        reader = pypdf.PdfReader(str(pdf_path))
        meta = reader.metadata
        if meta and getattr(meta, "title", None):
            t = meta.title.strip()
            if len(t) > 5 and not re.fullmatch(r"[\w\-]+", t):
                return t
        if reader.pages:
            text = reader.pages[0].extract_text() or ""
            lines = [l.strip() for l in text.splitlines() if l.strip()]
            for line in lines[:15]:
                if 8 <= len(line) <= 150 and re.search(r"[a-zA-Z]{3}", line):
                    skip = re.search(
                        r"^(abstract|introduction|doi|http|www|journal|vol\.?\s*\d|"
                        r"received|accepted|published|copyright|issn|isbn|page|\d+$)",
                        line, re.IGNORECASE)
                    if not skip:
                        return line
    except Exception:
        pass
    return pdf_path.stem

def extract_abstract(pdf_path, max_chars=800):
    import pypdf
    try:
        reader = pypdf.PdfReader(str(pdf_path))
        text = ""
        for page in reader.pages[:3]:
            text += page.extract_text() or ""
        m = re.search(r"abstract[\s\S]{0,20}?\n([\s\S]{50,800})", text, re.IGNORECASE)
        if m:
            return m.group(1).replace("\n", " ").strip()
        if reader.pages:
            return (reader.pages[0].extract_text() or "")[:max_chars]
    except Exception:
        pass
    return ""

def rename_pdf(pdf_path, title):
    safe = _sanitize_filename(title)
    if not safe:
        return pdf_path
    new_path = pdf_path.parent / (safe + ".pdf")
    if new_path == pdf_path:
        return pdf_path
    counter = 1
    while new_path.exists() and new_path != pdf_path:
        new_path = pdf_path.parent / (safe + f"_{counter}.pdf")
        counter += 1
    pdf_path.rename(new_path)
    return new_path

CLASSIFICATION_PROMPT = """You are a research librarian organizing a physics / materials science library.

Classify the document below into EXACTLY ONE of these 4 categories:

  subject_paper  - Research paper on transition metal chalcogenides (MxCy compounds:
                   oxides, sulfides, selenides, tellurides), electronic structure,
                   band gap, magnetic properties, DFT/GGA/LDA calculations, etc.

  physics_book   - Textbook or reference book on general physics topics
                   (solid state physics, quantum mechanics, thermodynamics, etc.)

  manuscript     - PhD thesis, dissertation, or master memoir.

  out_of_subject - Anything unrelated: biology, chemistry, machine learning, etc.

Respond with ONLY the category key. Nothing else.
Valid responses: subject_paper | physics_book | manuscript | out_of_subject

Title: {title}
Abstract: {abstract}
"""

VALID_CATS = {"subject_paper", "physics_book", "manuscript", "out_of_subject"}

_SUBJECT_KW    = ["chalcogenide","sulfide","selenide","telluride","oxide","band gap",
                  "magnetization","DFT","GGA","LDA","VASP","Wien2k","ab initio",
                  "first-principles","transition metal","half-metal","spin polariz",
                  "Hubbard","Mott","ferromagnet","antiferromagnet"]
_BOOK_KW       = ["textbook","introduction to","fundamentals","principles of",
                  "solid state physics","quantum mechanics","second edition",
                  "third edition","isbn","table of contents"]
_MANUSCRIPT_KW = ["thesis","dissertation","doctorat","phd","master","magistere",
                  "magister","memoire","submitted to","degree of doctor"]
_OUT_KW        = ["biology","polymer","organic","machine learning","deep learning",
                  "economics","finance","climate","medicine","pharmacol"]

def _keyword_classify(title, abstract):
    t = (title + " " + abstract).lower()
    scores = {
        "subject_paper":  sum(1 for kw in _SUBJECT_KW    if kw.lower() in t),
        "physics_book":   sum(1 for kw in _BOOK_KW       if kw.lower() in t),
        "manuscript":     sum(1 for kw in _MANUSCRIPT_KW if kw.lower() in t),
        "out_of_subject": sum(1 for kw in _OUT_KW        if kw.lower() in t),
    }
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "unclassified"

def classify(title, abstract, _retry=True):
    try:
        prompt = CLASSIFICATION_PROMPT.format(title=title[:300], abstract=abstract[:600])
        response = _gemini_client.generate_content(prompt)
        cat = response.text.strip().lower().replace(" ", "_")
        if cat in VALID_CATS:
            return cat, "gemini"
        for valid in VALID_CATS:
            if valid in cat:
                return valid, "gemini"
        return _keyword_classify(title, abstract), "keyword_fallback"
    except Exception as e:
        err = str(e)
        if _retry and ("429" in err or "quota" in err.lower()):
            print("  [!] Rate limit hit -- waiting 30s ...")
            time.sleep(30)
            return classify(title, abstract, _retry=False)
        print(f"  [!] Gemini error: {e} -- using keyword fallback")
        return _keyword_classify(title, abstract), "keyword_fallback"

CAT_FOLDERS = {
    "subject_paper":  "4_Subject_Papers",
    "physics_book":   "2_Physics_Books",
    "manuscript":     "3_Manuscripts",
    "out_of_subject": "1_Out_of_Subject",
    "unclassified":   "0_Unclassified",
}

CAT_COLORS = {
    "subject_paper":  "D9EAD3",
    "physics_book":   "FFF2CC",
    "manuscript":     "FCE5CD",
    "out_of_subject": "F4CCCC",
    "unclassified":   "EFEFEF",
}

def write_excel_report(records, output_path):
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    ws = wb.active
    ws.title = "All Papers"

    HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
    HEADER_FONT = Font(color="FFFFFF", bold=True)

    headers = ["#","Original Filename","New Title","Category",
               "Method","Folder","Abstract Snippet","Final Path"]
    ws.append(headers)
    for col in range(1, len(headers) + 1):
        cell = ws.cell(1, col)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center")

    for i, rec in enumerate(records, 1):
        cat    = rec.get("category", "unclassified")
        folder = CAT_FOLDERS.get(cat, CAT_FOLDERS["unclassified"])
        ws.append([
            i, rec["original_filename"], rec["title"], cat,
            rec.get("method", ""), folder, rec["abstract_snippet"][:200],
            rec.get("final_path", ""),
        ])
        fill = PatternFill("solid", fgColor=CAT_COLORS.get(cat, "FFFFFF"))
        for col in range(1, len(headers) + 1):
            ws.cell(i + 1, col).fill = fill

    for col, w in enumerate([5,45,70,20,18,30,60,60], 1):
        ws.column_dimensions[get_column_letter(col)].width = w

    ws2 = wb.create_sheet("Summary")
    ws2.append(["Category","Folder","Total","Gemini AI","Keyword fallback"])
    for cat, folder in CAT_FOLDERS.items():
        total  = sum(1 for r in records if r.get("category") == cat)
        ai_cnt = sum(1 for r in records if r.get("category") == cat and r.get("method") == "gemini")
        ws2.append([cat, folder, total, ai_cnt, total - ai_cnt])

    wb.save(str(output_path))

print("Functions defined.")

## Cell 6 — Step 1: Extract titles & rename

For a 1200-paper library this can take a while (PDF text extraction, not AI yet). Progress prints every file.

In [ ]:
pdf_files = sorted(paper_path.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s) in {paper_path}\n")

records = []
t0 = time.time()
for i, pdf in enumerate(pdf_files, 1):
    title    = extract_title(pdf)
    abstract = extract_abstract(pdf)
    new_pdf  = rename_pdf(pdf, title)
    records.append({
        "original_filename": pdf.name,
        "title":             title,
        "new_filename":      new_pdf.name,
        "abstract_snippet":  abstract[:200],
        "path":              new_pdf,
        "category":          None,
        "method":            None,
        "final_path":        "",
    })
    if i % 25 == 0 or i == len(pdf_files):
        elapsed = time.time() - t0
        rate = i / elapsed if elapsed > 0 else 0
        eta = (len(pdf_files) - i) / rate if rate > 0 else 0
        print(f"  [{i}/{len(pdf_files)}]  {100*i/len(pdf_files):.1f}%  |  {rate:.1f} files/s  |  ETA {eta:.0f}s")

print(f"\nDone. {len(records)} files processed in {time.time()-t0:.0f}s.")

## Cell 7 — Step 2: Classify with Gemini AI

This is the slow step — ~4.5 seconds per paper (free-tier rate limit). For 1200 papers, budget **~90 minutes**.

**Tip:** Colab free tier disconnects after long idle periods, but actively running code keeps the session alive. If it does disconnect partway through, see the resume cell further below.

In [ ]:
t0 = time.time()
for i, rec in enumerate(records, 1):
    cat, method = classify(rec["title"], rec["abstract_snippet"])
    rec["category"] = cat
    rec["method"]   = method
    tag = "[AI]" if method == "gemini" else "[KW]"
    if i % 10 == 0 or i == len(records):
        elapsed = time.time() - t0
        rate = i / elapsed if elapsed > 0 else 0
        eta_min = (len(records) - i) / rate / 60 if rate > 0 else 0
        print(f"  [{i}/{len(records)}] {tag} {rec['title'][:50]:50s} -> {cat}   (ETA {eta_min:.1f} min)")
    if method == "gemini" and i < len(records):
        time.sleep(RATE_LIMIT_SEC)

print("\nSummary:")
for cat, folder in CAT_FOLDERS.items():
    count = sum(1 for r in records if r.get("category") == cat)
    if count:
        print(f"  {folder:25s} {count:4d} file(s)")

## Cell 8 — Step 3: Move files into category subfolders

In [ ]:
for folder in CAT_FOLDERS.values():
    (paper_path / folder).mkdir(exist_ok=True)

errors = []
for rec in records:
    src    = paper_path / rec["new_filename"]
    folder = CAT_FOLDERS.get(rec["category"], CAT_FOLDERS["unclassified"])
    dst    = paper_path / folder / rec["new_filename"]
    counter = 1
    while dst.exists():
        stem = Path(rec["new_filename"]).stem
        dst  = paper_path / folder / f"{stem}_{counter}.pdf"
        counter += 1
    try:
        if src.exists():
            shutil.move(str(src), str(dst))
            rec["final_path"] = str(dst)
        else:
            rec["final_path"] = "FILE NOT FOUND"
            errors.append(rec["new_filename"])
    except Exception as e:
        rec["final_path"] = f"ERROR: {e}"
        errors.append(rec["new_filename"])

print(f"Moved {len(records) - len(errors)} files.")
if errors:
    print(f"{len(errors)} error(s):")
    for e in errors[:20]:
        print(f"  - {e}")

## Cell 9 — Export Excel report

In [ ]:
report_path = paper_path / "PDF_Library_Report.xlsx"
write_excel_report(records, report_path)

ai_count = sum(1 for r in records if r.get("method") == "gemini")
print(f"Done. {len(records)} files, {len(errors)} error(s)")
print(f"Gemini AI: {ai_count}   Keyword fallback: {len(records)-ai_count}")
print(f"Report saved to your Drive at: {report_path}")
print("\nYou can open it directly from Google Drive, or download it below.")

## Cell 10 (optional) — Download the report to your computer

In [ ]:
from google.colab import files
files.download(str(report_path))

---
## If your session disconnects partway through classification (Cell 7)

Colab free tier can disconnect after ~90 min of inactivity or ~12 hours total. If this happens during the slow classification step:

1. Re-run Cells 1-6 (dependencies, Drive mount, folder path, API key, functions, title extraction) — title extraction is fast and safe to redo.
2. Before re-running Cell 7, check which files are **already in a category subfolder** (meaning they were fully processed before disconnect) and skip those:

```python
# Paste this BEFORE cell 7 if resuming after a disconnect
already_done = set()
for folder in CAT_FOLDERS.values():
    folder_path = paper_path / folder
    if folder_path.exists():
        already_done.update(f.name for f in folder_path.glob('*.pdf'))

records = [r for r in records if r['new_filename'] not in already_done]
print(f'{len(already_done)} already processed, {len(records)} remaining')
```

Then continue with Cell 7 onward as normal — it will only classify the remaining files.